# Code Exercise 9

In [4]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.manifold import TSNE
import plotly_express as px

OHCO = ['author', 'book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']
BAG = OHCO[:5 ] # Paragraphs
window = 5

%matplotlib inline

In [5]:
TOKENS = pd.read_csv('TOKEN2.csv')
LIB = pd.read_csv('LIB.csv')
VOCAB = pd.read_csv('VOCAB2.csv')

TOKENS = TOKENS.merge(
    LIB[['book_id', 'author', 'title']],
    on='book_id',
    how='left'
)

TOKENS = TOKENS.set_index(OHCO)
TOKENS = TOKENS[~TOKENS.term_str.isna()]
TOKENS = TOKENS[~TOKENS.pos.str.match('NNPS?', na=False)]

In [6]:
TOKENS_RESET = TOKENS.reset_index()

CORPUS_DF = (
    TOKENS_RESET
    .groupby(BAG)['term_str']
    .apply(list)
    .reset_index()
)

In [7]:
austen_corpus = CORPUS_DF[CORPUS_DF['author'] == 'austen']['term_str'].tolist()
melville_corpus = CORPUS_DF[CORPUS_DF['author'] == 'melville']['term_str'].tolist()

In [8]:
austen_model = Word2Vec(
    sentences=austen_corpus,
    vector_size=100,
    window=window,
    min_count=10,
    workers=4,
    seed=23
)

melville_model = Word2Vec(
    sentences=melville_corpus,
    vector_size=100,
    window=window,
    min_count=10,
    workers=4,
    seed=23
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [9]:
austen_corpus = CORPUS_DF[CORPUS_DF['author'] == 'austen']['term_str'].tolist()
melville_corpus = CORPUS_DF[CORPUS_DF['author'] == 'melville']['term_str'].tolist()

In [10]:
austen_model = Word2Vec(
    sentences=austen_corpus,
    vector_size=100,
    window=window,
    min_count=10,
    workers=4,
    seed=23
)

melville_model = Word2Vec(
    sentences=melville_corpus,
    vector_size=100,
    window=window,
    min_count=10,
    workers=4,
    seed=23
)

In [13]:
def make_tsne_df(model, perplexity=40, random_state=23):
    words = model.wv.index_to_key
    vectors = np.array([model.wv[word] for word in words])

    tsne_model = TSNE(
        perplexity=perplexity,
        n_components=2,
        init='pca',
        max_iter=2500,
        random_state=random_state
    )

    tsne_values = tsne_model.fit_transform(vectors)

    tsne_df = pd.DataFrame({
        'word': words,
        'x': tsne_values[:, 0],
        'y': tsne_values[:, 1]
    })

    return tsne_df

In [44]:
austen_tsne = make_tsne_df(austen_model)
melville_tsne = make_tsne_df(melville_model)

In [15]:
fig_austen = px.scatter(
    austen_tsne,
    x='x',
    y='y',
    text='word',
    title='Austen Word Embeddings (t-SNE)',
    height=1000
)

fig_austen.update_traces(mode='text')
fig_austen.show()

In [16]:
fig_melville = px.scatter(
    melville_tsne,
    x='x',
    y='y',
    text='word',
    title='Melville Word Embeddings (t-SNE)',
    height=1000
)

fig_melville.update_traces(mode='text')
fig_melville.show()

## Austen t-SNE

The t-SNE visualization of Austen’s word embeddings shows several semantic regions. The t-SNE visualization of Austen’s word embeddings shows several meaningful semantic regions. One cluster contains words like observe, deny, judge, and agree. This region appears to reflect evaluation, interpretation, and social judgment, which fits Austen’s emphasis on interpersonal misunderstanding. Another cluster contains words like equal, address, and expectations which also fits Austen’s novels. Overall, Austen’s semantic space appears to be organized not just around domestic life, but also around judgment, communication, and social expectations.

## Melville Findings

The t-SNE visualization of Melville’s word embeddings reveals a few meaningful regions hidden in the larger cluster.  One clear cluster includes words such as sojourn, harbours, tracks, and cutters, which relate to travel, movement, and maritime activity. This reflects Melville’s strong focus on seafaring and exploration. Another cluster contains descriptive terms such as jagged, unsightly, tufted, miniature, and resplendent. These words suggest a semantic region centered on physical appearance and detailed visual description, which aligns with Melville’s rich and descriptive prose style. Overall, Melville’s semantic space appears to combine themes of travel and maritime life with detailed physical description and emotional complexity.

In [34]:
def analogy(model, A, B, C, n=5):
    try:
        return model.wv.most_similar(
            positive=[B, C],
            negative=[A],
            topn=n
        )
    except KeyError as e:
        print("Missing word:", e)

In [35]:
analogy(austen_model, 'brother', 'sister', 'son')


[('daughter', 0.9097371697425842),
 ('wife', 0.863010048866272),
 ('choice', 0.838087260723114),
 ('attentions', 0.831770658493042),
 ('friend', 0.8292211890220642)]

In [36]:
analogy(austen_model, 'daughter', 'son', 'mother')


[('brother', 0.9049022197723389),
 ('uncle', 0.9034225940704346),
 ('father', 0.8968230485916138),
 ('sister', 0.8952520489692688),
 ('aunt', 0.8877330422401428)]

In [37]:
analogy(austen_model, 'boy', 'girl', 'man')

[('woman', 0.8711153864860535),
 ('person', 0.6188589930534363),
 ('young', 0.5978150367736816),
 ('lady', 0.597022294998169),
 ('sensible', 0.5789365172386169)]

In [38]:
analogy(melville_model, 'ship', 'boat', 'captain')


[('mate', 0.8424328565597534),
 ('doctor', 0.817966103553772),
 ('chief', 0.7951207756996155),
 ('stranger', 0.7478972673416138),
 ('steward', 0.737297534942627)]

In [39]:
analogy(melville_model, 'captain', 'mate', 'ship')


[('boat', 0.839260458946228),
 ('frigate', 0.7669917941093445),
 ('cabin', 0.7523902058601379),
 ('main', 0.7442272305488586),
 ('vessel', 0.7435215711593628)]

In [40]:
analogy(melville_model, 'whale', 'ship', 'fish')

[('shores', 0.7710114121437073),
 ('quarters', 0.7319257259368896),
 ('sailing', 0.7277123928070068),
 ('ropes', 0.7269542813301086),
 ('hammocks', 0.7260351777076721)]

## Analogies

The Austen model produced very strong analogies, particularly in the domain of family and gender relationships like 'brother, sister, son' returned daughter as the top result with a high similarity score. Similarly, 'boy, girl, man' correctly returned woman. These results demonstrate that the model effectively captures structured relationships such as gender and kinship, which are central themes in Austen’s novels. The Melville model also produced meaningful analogies, though they were more domain-specific. For example, the analogy 'ship, boat, captain' correctly returned mate, reflecting the hierarchical structure of roles aboard a ship. Another analogy, 'captain, mate, ship', returned terms such as boat, frigate, and vessel, showing that the model captures relationships within the maritime domain. More complex analogies were less precise but still remained within the general context of seafaring. Overall, the analogies demonstrate that word embeddings capture meaningful semantic relationships, with Austen’s model excelling in social structures and Melville’s model reflecting maritime themes.